# 탐지 모델 개선 실험 — A: 신뢰도 기준 다시 정하기 / B: YOLO 추가 학습

**위에서부터 `Shift + Enter`만 누르면 됩니다. 고칠 값 없음** (고르기 자동, 이미 끝난 단계는 건너뜀).

가상 염색은 `run_unsb_finish`에서 고른 UNSB 설정(`07. Evaluate/3_staining/unsb/selected.json`, 현재 ep15 · NFE 1)을 그대로 씁니다.

| 실험 | 탐지 모델 | 바꾸는 것 |
| --- | --- | --- |
| 지금까지 | YOLO (진짜 염색으로만 학습) | conf 0.7 고정 (염색 val로 정한 값) |
| **A** | 같은 YOLO | conf를 **가상 염색 val**로 다시 고름 (0.5~0.9) |
| **B** | YOLO를 **가상 염색한 RedTell 학습 영상(dse 1~83) + TXL**로 추가 학습 | conf도 가상 염색 val로 고름 |

- B는 원래 계획("탐지 모델은 진짜 염색으로만 학습")을 바꾼 추가 실험입니다. 보고서에는 별도 실험으로 적습니다.
- 고르는 데는 val 9장만, test 91장은 모델마다 한 번만 채점합니다.
- B 모델이 진짜 염색 영상 실력을 잊지 않았는지 TXL test로도 확인합니다.

예상 시간: 1번 수 분, 2번(추가 학습) 30분~1시간, 3~4번 수 분. 결과: `07. Evaluate/6_detector/`

In [1]:
import torch, os, sys, json, pandas as pd
print('GPU:', torch.cuda.is_available())
os.chdir(r'D:\05. Blood\05. Train\code')
P = r"D:/05. Blood/04. Prep"; T = r"D:/05. Blood/05. Train"
UREPO = T + "/external/unsb"
sel = json.load(open("D:/05. Blood/07. Evaluate/3_staining/unsb/selected.json"))
BE, BK = sel["epoch"], sel["nfe"]
VS = f"D:/05. Blood/06. Generate/unsb/ep{BE}_nfe{BK}"
R6 = "D:/05. Blood/07. Evaluate/6_detector"
YOLO_BASE = T + "/runs/yolo/yolo11s/weights/best.pt"
YOLO_FT = T + "/runs/yolo/yolo11s_ft_unsb/weights/best.pt"
print(f"가상 염색: UNSB ep{BE}, NFE {BK}  →  {VS}")

GPU: True
가상 염색: UNSB ep15, NFE 1  →  D:/05. Blood/05. Train/virtual_stained/unsb/ep15_nfe1


## 1. (B 준비) RedTell 학습 영상 + 정답 만들기 → UNSB로 가상 염색

In [2]:
!{sys.executable} "{P}/code/07_make_dse_train.py" --prep "{P}" --dse_zip "D:/05. Blood/03. Data/6. RedTell/dse_data.zip"
n_done = len(os.listdir(VS + "/train")) if os.path.isdir(VS + "/train") else 0
if n_done < 82:
    !{sys.executable} 23_apply_unsb.py --repo "{UREPO}" --ckpt "{T}/runs/unsb/unsb_rbc" --epoch {BE} --nfe {BK} --images "{P}/6_redtell_dse_train/images" --out "{VS}/train"

[학습용] D:/05. Blood/04. Prep\6_redtell_dse_train : 영상 82장, 정답 RBC 2992개
UNSB 가상 염색 82장 완료 (epoch 15, NFE 1, 조각 128, 장당 1096 ms) → D:/05. Blood/05. Train/virtual_stained/unsb/ep15_nfe1/train


## 2. (B) YOLO 추가 학습 — 30 epoch, 진짜 염색으로 학습한 YOLO에서 시작

In [3]:
if not os.path.exists(YOLO_FT):
    !{sys.executable} 24_finetune_yolo.py --prep "{P}" --train_root "{T}" --vs_train "{VS}/train" --vs_val "{VS}/val" --name yolo11s_ft_unsb --epochs 30

�뜲�씠�꽣: 媛��긽 �뿼�깋 �븰�뒿 82�옣 횞 4 + TXL �븰�뒿 �쟾泥�, 寃�利� 9�옣 �넂 D:/05. Blood/05. Train\datasets\yolo_ft_yolo11s_ft_unsb\data.yaml
New https://pypi.org/project/ultralytics/8.4.203 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.184  Python-3.11.16 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16283MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/05. Blood/05. Train\datasets\yolo_ft_yolo11s_ft_unsb\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None,

## 3. (A·B) 신뢰도 기준을 가상 염색 val로 고르기 — 규칙: val RBC F1 최고

In [4]:
CONFS = [0.5, 0.6, 0.7, 0.8, 0.9]
MODELS = {"base": YOLO_BASE, "ft": YOLO_FT}
for m, w in MODELS.items():
    for c in CONFS:
        out = f"{R6}/{m}_conf{c}_val"
        if not os.path.exists(out + "/summary.csv"):
            !{sys.executable} 12_eval_compare.py --prep "{P}" --images "{VS}/val" --labels "{P}/4_eval_unstained/redtell_dse_val/labels" --yolo "{w}" --conf {c} --out "{out}"
rows = []
for m in MODELS:
    for c in CONFS:
        d = pd.read_csv(f"{R6}/{m}_conf{c}_val/summary.csv").iloc[0]
        rows.append({"model": m, "conf": c, "RBC_MAE": d.RBC_MAE, "RBC_precision": d.RBC_precision, "RBC_recall": d.RBC_recall, "RBC_F1": d.RBC_F1})
vt = pd.DataFrame(rows); display(vt)
BEST = {m: float(vt[vt.model == m].sort_values(["RBC_F1", "RBC_MAE"], ascending=[False, True]).iloc[0].conf) for m in MODELS}
print("val로 고른 conf:", BEST)
json.dump(BEST, open(R6 + "/selected_conf.json", "w"))


[YOLO]
  method: YOLO
  n_images: 9
  WBC_MAE: 0.0
  WBC_precision: 0.0
  WBC_recall: 0.0
  WBC_F1: 0.0
  RBC_MAE: 11.111
  RBC_precision: 0.7192
  RBC_recall: 0.932
  RBC_F1: 0.8119
  WBC_rel_error: 
  RBC_rel_error: 0.3946
  ms_per_image: 9.5

����옣: D:/05. Blood/05. Train/results/6_detector/base_conf0.5_val

[YOLO]
  method: YOLO
  n_images: 9
  WBC_MAE: 0.0
  WBC_precision: 0.0
  WBC_recall: 0.0
  WBC_F1: 0.0
  RBC_MAE: 7.889
  RBC_precision: 0.7628
  RBC_recall: 0.9231
  RBC_F1: 0.8353
  WBC_rel_error: 
  RBC_rel_error: 0.2937
  ms_per_image: 10.1

����옣: D:/05. Blood/05. Train/results/6_detector/base_conf0.6_val

[YOLO]
  method: YOLO
  n_images: 9
  WBC_MAE: 0.0
  WBC_precision: 0.0
  WBC_recall: 0.0
  WBC_F1: 0.0
  RBC_MAE: 4.444
  RBC_precision: 0.8165
  RBC_recall: 0.9083
  RBC_F1: 0.8599
  WBC_rel_error: 
  RBC_rel_error: 0.1651
  ms_per_image: 8.5

����옣: D:/05. Blood/05. Train/results/6_detector/base_conf0.7_val

[YOLO]
  method: YOLO
  n_images: 9
  WBC_MAE: 0.0
  WBC_pr

,model,conf,RBC_MAE,RBC_precision,RBC_recall,RBC_F1
0,base,0.5,11.111,0.7192,0.9320,0.8119
1,base,0.6,7.889,0.7628,0.9231,0.8353
2,base,0.7,4.444,0.8165,0.9083,0.8599
3,base,0.8,2.444,0.8618,0.8669,0.8643
4,base,0.9,7.667,0.9524,0.7692,0.8511
5,ft,0.5,3.667,0.9057,0.9941,0.9478
6,ft,0.6,2.667,0.9171,0.9822,0.9486
7,ft,0.7,2.000,0.9422,0.9645,0.9532
8,ft,0.8,2.111,0.9657,0.9172,0.9408
9,ft,0.9,14.222,1.0000,0.6213,0.7664


val로 고른 conf: {'base': 0.8, 'ft': 0.7}


## 4. test 한 번씩 + 최종 비교표

In [5]:
runs = {"A: 원래 YOLO, conf 다시 고름": ("base", f"{VS}/test", "redtell_dse"),
        "B: 추가 학습 YOLO": ("ft", f"{VS}/test", "redtell_dse"),
        "B 모델을 진짜 염색 TXL test에 (잊지 않았는지)": ("ft", None, "txl")}
for name, (m, img, kind) in runs.items():
    c = BEST[m]; w = MODELS[m]
    out = f"{R6}/{m}_conf{c}_{'test' if kind == 'redtell_dse' else 'txl_test'}"
    if os.path.exists(out + "/summary.csv"):
        continue
    if kind == "redtell_dse":
        !{sys.executable} 12_eval_compare.py --prep "{P}" --images "{img}" --labels "{P}/4_eval_unstained/redtell_dse/labels" --yolo "{w}" --conf {c} --out "{out}"
    else:
        !{sys.executable} 12_eval_compare.py --prep "{P}" --yolo "{w}" --conf {c} --out "{out}"
cols = ["RBC_MAE", "RBC_precision", "RBC_recall", "RBC_F1", "WBC_F1"]
tab = {
  "[참고 목표] 진짜 염색 TXL test, 원래 YOLO conf 0.7": "D:/05. Blood/07. Evaluate/1_detection/eval0_yolo_conf0.7",
  "비염색 그대로 (하한)": "D:/05. Blood/07. Evaluate/2_no_staining/stain_none_test",
  f"UNSB ep{BE} NFE{BK} + 원래 YOLO conf 0.7": f"D:/05. Blood/07. Evaluate/3_staining/unsb/ep{BE}_nfe{BK}_test",
  f"A: UNSB + 원래 YOLO conf {BEST['base']}": f"{R6}/base_conf{BEST['base']}_test",
  f"B: UNSB + 추가 학습 YOLO conf {BEST['ft']}": f"{R6}/ft_conf{BEST['ft']}_test",
  f"B 모델, 진짜 염색 TXL test (conf {BEST['ft']})": f"{R6}/ft_conf{BEST['ft']}_txl_test",
}
final = pd.DataFrame({k: pd.read_csv(v + "/summary.csv").iloc[0][cols] for k, v in tab.items()}).T
display(final)
final.to_csv(R6 + "/final_compare.csv", encoding="utf-8-sig")


[YOLO]
  method: YOLO
  n_images: 91
  WBC_MAE: 0.0
  WBC_precision: 0.0
  WBC_recall: 0.0
  WBC_F1: 0.0
  RBC_MAE: 3.516
  RBC_precision: 0.8149
  RBC_recall: 0.8437
  RBC_F1: 0.829
  WBC_rel_error: 
  RBC_rel_error: 0.1232
  ms_per_image: 10.9

����옣: D:/05. Blood/05. Train/results/6_detector/base_conf0.8_test

[YOLO]
  method: YOLO
  n_images: 91
  WBC_MAE: 0.0
  WBC_precision: 0.0
  WBC_recall: 0.0
  WBC_F1: 0.0
  RBC_MAE: 2.484
  RBC_precision: 0.9385
  RBC_recall: 0.969
  RBC_F1: 0.9535
  WBC_rel_error: 
  RBC_rel_error: 0.0605
  ms_per_image: 10.2

����옣: D:/05. Blood/05. Train/results/6_detector/ft_conf0.7_test

[YOLO]
  method: YOLO
  n_images: 126
  WBC_MAE: 0.008
  WBC_precision: 0.9924
  WBC_recall: 1.0
  WBC_F1: 0.9962
  RBC_MAE: 1.317
  RBC_precision: 0.916
  RBC_recall: 0.9496
  RBC_F1: 0.9325
  WBC_rel_error: 0.004
  RBC_rel_error: 0.1086
  ms_per_image: 11.4

����옣: D:/05. Blood/05. Train/results/6_detector/ft_conf0.7_txl_test


,RBC_MAE,RBC_precision,RBC_recall,RBC_F1,WBC_F1
"[참고 목표] 진짜 염색 TXL test, 원래 YOLO conf 0.7",1.206,0.9129,0.9627,0.9371,1.0
비염색 그대로 (하한),9.846,0.9736,0.7252,0.8313,0.0
UNSB ep15 NFE1 + 원래 YOLO conf 0.7,6.231,0.7626,0.881,0.8175,0.0
A: UNSB + 원래 YOLO conf 0.8,3.516,0.8149,0.8437,0.829,0.0
B: UNSB + 추가 학습 YOLO conf 0.7,2.484,0.9385,0.969,0.9535,0.0
"B 모델, 진짜 염색 TXL test (conf 0.7)",1.317,0.916,0.9496,0.9325,0.9962
